# RQ3: comparing the models

Same monthly panel, but here I put the lag regression up against a plain seasonal-naive rule (repeat the value from a year ago) on the same 12-month test window.

## Setup (local or Google Colab)

On Colab this pulls the project data and code from GitHub. Locally it uses the repo folder you already have.

In [1]:
# Works whether you run this locally or on Google Colab.
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.isdir('data'):
    import subprocess
    subprocess.run(['git','clone','-q','https://github.com/bcudjoe/mobile-money-ghana-forecasting.git'])
    os.chdir('mobile-money-ghana-forecasting')
print('Colab' if IN_COLAB else 'local')

local


In [2]:
import pandas as pd, numpy as np
m=pd.read_csv('data/processed/monthly_series_clean.csv', parse_dates=['date'])
d=m.dropna(subset=['mm_value_lag1','mm_value_lag12','mm_value_roll3']).reset_index(drop=True)
Xc=['mm_value_lag1','mm_value_lag12','mm_value_roll3','agent_density','account_ownership','inflation','policy_rate','exch_rate','elevy','t_index']
X=d[Xc].values.astype(float); y=d['mm_value'].values.astype(float); H=12
Xtr,Xte=X[:-H],X[-H:]; ytr,yte=y[:-H],y[-H:]
mu=Xtr.mean(0); sd=Xtr.std(0); sd[sd==0]=1
beta,*_=np.linalg.lstsq(np.c_[np.ones(len(Xtr)),(Xtr-mu)/sd],ytr,rcond=None)
ols=np.c_[np.ones(len(Xte)),(Xte-mu)/sd]@beta
snaive=d['mm_value_lag12'].values[-H:]

## Side-by-side error metrics

In [3]:
def metrics(y,p):
    e=y-p; return dict(RMSE=np.sqrt(np.mean(e**2)), MAE=np.mean(np.abs(e)), MAPE=np.mean(np.abs(e/y))*100, R2=1-np.sum(e**2)/np.sum((y-y.mean())**2))
res=pd.DataFrame({'Lag-regression':metrics(yte,ols),'Seasonal-naive':metrics(yte,snaive)}).T
res=res.round({'RMSE':0,'MAE':0,'MAPE':2,'R2':3})
print(res.to_string())
print('\nBest model by MAPE:', res['MAPE'].idxmin())

                    RMSE       MAE   MAPE     R2
Lag-regression   23396.0   19574.0   5.43  0.821
Seasonal-naive  131006.0  127369.0  33.87 -4.620

Best model by MAPE: Lag-regression


## What comes next

The final report adds SARIMA, Prophet, and gradient boosting and settles the comparison with a Diebold-Mariano test. The block below runs SARIMA only if statsmodels is installed, so the notebook still works without it.

In [4]:
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    ytr_s = pd.Series(ytr)
    sar = SARIMAX(ytr_s, order=(1,1,1), seasonal_order=(1,0,0,12)).fit(disp=False)
    sar_pred = sar.forecast(H).values
    print('SARIMA MAPE:', round(np.mean(np.abs((yte-sar_pred)/yte))*100,2))
except Exception as ex:
    print('statsmodels not installed; SARIMA is a planned next step. (', type(ex).__name__, ')')

statsmodels not installed; SARIMA is a planned next step. ( ModuleNotFoundError )
